### Import modules

In [14]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout

# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


### Load Data

In [15]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_file_level_embeddings.csv')
# data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/embeddings/Gemini_embeddings.csv')

In [16]:
data = data.drop(columns=['file'])
data

,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,...,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767,dim_768,pattern
0,-0.028630,0.014242,-0.084164,0.006628,0.054243,0.067711,0.027371,-0.006613,-0.034239,-0.014033,...,0.020441,0.016043,0.042599,0.010227,0.028914,-0.026917,-0.046186,0.094621,-0.040274,Advanced LLM Prompting
1,-0.005000,0.003826,-0.077749,-0.017940,0.028686,0.041942,0.004677,0.013499,-0.024628,-0.017825,...,0.016375,0.028125,0.043700,-0.006213,0.003327,-0.021241,0.000565,0.114473,-0.067262,Advanced LLM Prompting
2,-0.030237,-0.001334,-0.048906,0.036134,0.053802,0.039611,0.050056,-0.000391,-0.026402,-0.024241,...,0.052654,-0.022873,0.049713,0.004830,-0.002995,-0.079745,-0.064036,0.085545,-0.041079,Advanced LLM Prompting
3,-0.027586,-0.001820,-0.042761,0.028486,0.049057,0.025219,0.053730,-0.004445,-0.007426,-0.026629,...,0.030581,-0.001082,0.034104,-0.003824,0.007942,-0.041021,-0.074196,0.076562,-0.045155,Advanced LLM Prompting
4,0.000132,0.018428,0.008781,0.018593,0.068425,0.036489,0.047712,-0.042397,-0.009086,-0.019750,...,0.044203,-0.020275,0.014996,-0.010121,0.007810,-0.054733,-0.059098,0.067748,-0.042832,Advanced LLM Prompting
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1741,-0.002423,0.005586,-0.078116,0.031092,0.051569,0.034309,0.019771,0.018993,-0.024791,-0.013303,...,0.018675,0.012948,0.032034,-0.026418,-0.020808,-0.007870,-0.035148,0.088527,-0.049821,Tool Use for LLMs
1742,0.001156,0.003813,-0.026644,0.010744,0.048009,0.069874,0.024441,-0.001617,0.018773,0.043720,...,0.031216,-0.039242,0.054849,0.016675,0.031247,-0.012921,-0.068411,0.049576,-0.032782,Tool Use for LLMs
1743,0.013422,-0.023461,-0.058135,0.031028,0.072383,0.046020,0.054596,0.028204,-0.025168,0.006556,...,-0.007320,-0.019408,0.021146,-0.007970,0.010736,-0.048937,-0.045026,0.049606,-0.008026,Tool Use for LLMs
1744,-0.011074,-0.000204,-0.057851,-0.028101,0.009391,0.004398,-0.012365,0.016093,-0.059569,-0.032780,...,0.029659,0.021206,0.024620,-0.016576,-0.006691,-0.012738,0.016579,0.099074,-0.070548,Tool Use for LLMs


In [ ]:
# from sklearn.decomposition import PCA
# pca = PCA(n_components=700)
# # data = data.drop(columns=['file'])
# label = 'label'
# dx = pca.fit_transform(data.drop(columns=[label]))
# lc = data[label]
# data = pd.DataFrame(dx)
# data['pattern'] = lc

### NN Architecture

In [17]:
TARGET_COLUMN = "pattern"

ARTIFACT_DIR = Path("../models/pattern_nn_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "pattern_classifier.keras"
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"
ENCODER_PATH = ARTIFACT_DIR / "label_encoder.joblib"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_features:
    raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

X = data[numeric_features].fillna(0.0).values
y = data[TARGET_COLUMN].astype(str).values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

X_train, X_temp, y_train_enc, y_temp_enc = train_test_split(
    X,
    y_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_encoded,
)
X_val, X_test, y_val_enc, y_test_enc = train_test_split(
    X_temp,
    y_temp_enc,
    test_size=0.5,
    random_state=42,
    stratify=y_temp_enc,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)

def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            Dense(num_classes, activation="softmax"),
        ]
    )

model = build_classifier(X_train.shape[1], num_classes)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
)

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=96,
    callbacks=callbacks,
    verbose=1,
)

test_loss, test_acc, test_top3 = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

y_pred = model.predict(X_test)
y_pred_labels = y_pred.argmax(axis=1)
report = classification_report(
    y_test_enc,
    y_pred_labels,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
class_breakdown = (
    report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
)

print("\nKey metrics:")
display(summary)
print("\nTop classes by support:")
display(class_breakdown)

# Persist artifacts for downstream inference pipelines
model.save(MODEL_PATH, include_optimizer=True)
joblib.dump(scaler, SCALER_PATH)
joblib.dump(label_encoder, ENCODER_PATH)
metadata = {
    "target_column": TARGET_COLUMN,
    "numeric_features": numeric_features,
    "num_classes": num_classes,
    "label_classes": label_encoder.classes_.tolist(),
    "test_metrics": {
        "loss": float(test_loss),
        "accuracy": float(test_acc),
        "top3_accuracy": float(test_top3),
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2))
print(f"Saved model to {MODEL_PATH}")
print(f"Saved scaler to {SCALER_PATH}")
print(f"Saved label encoder to {ENCODER_PATH}")
print(f"Saved metadata to {METADATA_PATH}")


Epoch 1/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.2636 - loss: 2.8335 - top3_acc: 0.4439 - val_accuracy: 0.5092 - val_loss: 1.9354 - val_top3_acc: 0.6881 - learning_rate: 0.0010
Epoch 2/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.2636 - loss: 2.8335 - top3_acc: 0.4439 - val_accuracy: 0.5092 - val_loss: 1.9354 - val_top3_acc: 0.6881 - learning_rate: 0.0010
Epoch 2/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.5867 - loss: 1.6650 - top3_acc: 0.7647 - val_accuracy: 0.5963 - val_loss: 1.5483 - val_top3_acc: 0.8165 - learning_rate: 0.0010
Epoch 3/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.5867 - loss: 1.6650 - top3_acc: 0.7647 - val_accuracy: 0.5963 - val_loss: 1.5483 - val_top3_acc: 0.8165 - learning_rate: 0.0010
Epoch 3/200
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.6875 - loss: 1.2295 - top3_acc: 0.8648 - val_accuracy: 0.6789 - val_loss: 1.2537 - val_top3_acc: 0.8670 - learning_rate: 0.0010
Epoch 4/200
14/14 ━━━━━━━━━━

,precision,recall,f1-score,support
accuracy,0.772,0.772,0.772,0.772
macro avg,0.756,0.769,0.757,219.000
weighted avg,0.759,0.772,0.760,219.000



Top classes by support:


,precision,recall,f1-score,support
LLM Code Execution for Precision,0.867,1.000,0.929,13.0
LLM Context Management,1.000,0.923,0.960,13.0
LLM based Multimodal Generative Prompting,0.929,1.000,0.963,13.0
Advanced LLM Prompting,0.545,0.500,0.522,12.0
Cross-lingual LLM Prompting,0.857,1.000,0.923,12.0
Enhanced User Intent Comprehension with LLMs,1.000,0.917,0.957,12.0
LLM Agent Training & Alignment,1.000,0.667,0.800,12.0
Integrating External Knlowladge with LLM,0.733,0.917,0.815,12.0


Saved model to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/pattern_classifier.keras
Saved scaler to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/scaler.joblib
Saved label encoder to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/label_encoder.joblib
Saved metadata to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/metadata.json


In [ ]:
def load_pattern_classifier(
    model_path: Path = MODEL_PATH,
    scaler_path: Path = SCALER_PATH,
    encoder_path: Path = ENCODER_PATH,
):
    """Load the saved NN, scaler, and label encoder for inference."""
    model = tf.keras.models.load_model(model_path)
    scaler = joblib.load(scaler_path)
    encoder = joblib.load(encoder_path)
    return model, scaler, encoder


def predict_pattern_probabilities(
    embeddings: np.ndarray,
    model: tf.keras.Model,
    scaler: StandardScaler,
    encoder: LabelEncoder,
) -> pd.DataFrame:
    """Return class probability dataframe for the provided embeddings."""
    embeddings = np.atleast_2d(embeddings)
    scaled = scaler.transform(embeddings)
    probs = model.predict(scaled)
    return pd.DataFrame(probs, columns=encoder.classes_)


# Example usage (uncomment to run):
# loaded_model, loaded_scaler, loaded_encoder = load_pattern_classifier()
# sample_probs = predict_pattern_probabilities(X_test[:5], loaded_model, loaded_scaler, loaded_encoder)
# display(sample_probs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 768)            │       538,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 768)            │         3,072 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 18)             │         2,322 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,308,216 (12.62 MB)

 Trainable params: 1,101,714 (4.20 MB)

 Non-trainable params: 3,072 (12.00 KB)

 Optimizer params: 2,203,430 (8.41 MB)

### Logistic Regression classifier

In [ ]:
# Logistic regression stage intentionally skipped per latest workflow requirements.
# The end-to-end classifier now relies solely on the neural network above.


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogReg test accuracy: 0.7943 | Top-3 accuracy: 0.9286

Logistic Regression summary metrics:


,precision,recall,f1-score,support
accuracy,0.794,0.794,0.794,0.794
macro avg,0.800,0.794,0.793,350.000
weighted avg,0.800,0.794,0.793,350.000



Per-class breakdown:


,precision,recall,f1-score,support
LLM Results Evaluation,0.765,0.650,0.703,20.0
Integrating External Knlowladge with LLM,0.944,0.850,0.895,20.0
"Reliable, Transparent, & Augmented LLMs",0.550,0.550,0.550,20.0
Tool Use for LLMs,0.545,0.600,0.571,20.0
LLM Context Management,0.680,0.850,0.756,20.0
LLM Code Execution for Precision,0.950,0.950,0.950,20.0
LLMs for Recommender Systems,0.850,0.850,0.850,20.0
LLM based Multimodal Generative Prompting,1.000,1.000,1.000,20.0
Cross-lingual LLM Prompting,0.947,0.947,0.947,19.0
Advanced LLM Prompting,0.688,0.579,0.629,19.0
